In [1]:
import tqdm

import numpy as np
import pandas as pd
import geopandas as gpd

import networkx as nx

from utils_code.utils_basic import PROJECT_PATH, SG_PROJECTED_CRS
from utils_code.utils_loaddata import read_zip_csv

# 1. Load data

In [2]:
zippath = PROJECT_PATH / 'data/Singapore/bus_network/raw_data/bus_network_2024-09-19.zip'

# Bus location
bus_loc = read_zip_csv(str(zippath) + '!bus_stop_2024-09-19.csv',
        dtype = {'BusStopCode': str}) \
    .rename(columns = {
        'BusStopCode'  : 'bus_stop_code',
        'RoadName'     : 'road_name',
        'Description'  : 'stop_name',
        'Latitude'     : 'lat',
        'Longitude'    : 'lon'})

# to GeoDataFrame
bus_loc = gpd.GeoDataFrame(
        bus_loc, 
        geometry = gpd.points_from_xy(x=bus_loc['lon'], y=bus_loc['lat']),
        crs = 'EPSG:4326') \
    .to_crs(SG_PROJECTED_CRS) \
    .assign(
        x = lambda g: g['geometry'].x, 
        y = lambda g: g['geometry'].y)
print('\nShape of bus location:', bus_loc.shape,
      '\nNumber of bus stop:', bus_loc['bus_stop_code'].nunique(),)


# Bus route
bus_route = read_zip_csv(str(zippath) + '!bus_route_2024-09-19.csv',
        dtype={
            'BusStopCode': str,
            'ServiceNo': str,
            'StopSequence' : int}) \
    .rename(columns={
        'Operator'     : 'bus_operator',
        'BusStopCode'  : 'bus_stop_code',
        'ServiceNo'    : 'bus_service_no',
        'StopSequence' : 'order_no',
        'Direction'    : 'route_direction',
        'Distance'     : 'stop_spacing' })
print('\nShape of bus route:', bus_route.shape)

# bus_service = pd.read_csv('original_data/bus_service_2024-09-19.csv',
#                           dtype={'BusStopCode': str, 'OriginCode': str, 'DestinationCode' : str})


Shape of bus location: (5136, 8) 
Number of bus stop: 5136

Shape of bus route: (25450, 12)


# 2. Create bus network

In [3]:
from typing import List

def node_sequence_to_edge_list(
    data: pd.DataFrame,
    id_cols: List[str],
    seq_col: str
) -> pd.DataFrame:
    """
    Convert ordered node sequences into an edge list of consecutive pairs.

    For each group defined by `id_cols`, this function:
      1. Sorts rows by `seq_col`
      2. Emits one edge between each consecutive pair of `node_id_col`
         as `left_{node_id_col}` → `right_{node_id_col}`

    Parameters
    ----------
    data : pd.DataFrame
        Input table. Must contain the grouping columns (`id_cols`),
        the node identifier column (`node_id_col`), and the sequence
        ordering column (`seq_col`).
    id_cols : list of str
        Column name(s) to group by (e.g., trip IDs, user IDs).
    seq_col : str
        Name of the column that defines the ordering within each group.

    Returns
    -------
    pd.DataFrame
        Edge list with columns:
          - the grouping columns in `id_cols`
          - `left_{node_id_col}`, `right_{node_id_col}` for each edge
    """
    edges: List[pd.DataFrame] = []

    for group_keys, group_df in tqdm.tqdm(data.groupby(by=id_cols, as_index=True), desc='Iterating bus line...'):
        # Ensure group_keys is a tuple
        if not isinstance(group_keys, tuple):
            group_keys = (group_keys, )

        # Sort by the sequence column
        group_df = (
            group_df
            .copy()
            .sort_values(by = seq_col, ascending = True, ignore_index = True)
            .drop(columns = id_cols + [seq_col])
        )

        # Need at least two nodes to form an edge
        if len(group_df) < 2:
            continue

        # Build the left/right node series
        left_df  = (
            group_df
            .copy()
            .iloc[:-1, :]
            .add_prefix('left_')
            .reset_index(drop=True)
        )
        right_df = (
            group_df
            .copy()
            .iloc[1:, :]
            .add_prefix('right_')
            .reset_index(drop=True)
        )

        edge_df = pd.concat([left_df, right_df], axis=1, ignore_index=False)

        # Add the grouping key columns back
        for col_name, key_val in zip(id_cols, group_keys):
            edge_df[col_name] = key_val

        edges.append(edge_df)

    edge_list = pd.concat(edges, ignore_index=True)

    return edge_list
# =============================================================================================

In [4]:
use_cols = ['bus_operator', 'bus_service_no', 'route_direction', 'order_no', 'bus_stop_code', 'stop_spacing']

# Convert bus stop sequences into edge list
edge_list = node_sequence_to_edge_list(
    data = bus_route[use_cols],
    id_cols = ['bus_operator', 'bus_service_no', 'route_direction'],
    seq_col = 'order_no')

# Compute stop spacing
edge_list = edge_list.assign(
        length_m = lambda df: (df['right_stop_spacing'] - df['left_stop_spacing']) * 1000) \
    .drop(columns=  ['left_stop_spacing', 'right_stop_spacing']) \
    .rename( columns= {
        'left_bus_stop_code': 'source',
        'right_bus_stop_code': 'target'})

print('\nNegative stop spacing:\n', edge_list[edge_list['length_m'] < 0])



# Aggregate edge attributes
join_unique = lambda x: ','.join(sorted(map(str, set(x))))

edge_list = (edge_list
    .query('length_m > 0')
    .groupby(by = ['source', 'target'], as_index = False)
    .agg(
        length_m     = ('length_m', 'mean'),
        length_m_max = ('length_m', 'max'),
        length_m_min = ('length_m', 'min'),
        length_m_std = ('length_m', 'std'),
        line_frequency = ('bus_service_no', 'count'),
        bus_operator    = ('bus_operator', join_unique),
        bus_service_no  = ('bus_service_no', join_unique),
        route_direction = ('route_direction', join_unique),
    )
)

Iterating bus line...: 100%|██████████| 721/721 [00:01<00:00, 433.20it/s]



Negative stop spacing:
       source target bus_operator bus_service_no  route_direction  length_m
23272  58991  59009          TTS            857                1  -42300.0


In [5]:
net = nx.DiGraph()
# add nodes from dataframe
net.add_nodes_from(
    bus_loc
    .drop(columns=['lat', 'lon', 'geometry'])
    .set_index('bus_stop_code', verify_integrity=True)
    .to_dict(orient='index')
    .items()
)


# Add edges from edge list
edge_list = edge_list.drop(columns=['length_m_max', 'length_m_min', 'length_m_std'])

net.add_edges_from([(
    r['source'], r['target'],
    r.drop(labels=['source', 'target']).to_dict()
    ) for _, r in edge_list.iterrows()
])

# Save bus network graph
nx.write_gexf(net, PROJECT_PATH / 'bus_network_2024-09-19.gexf')

# ----Previous code to add edge attributes----

In [6]:
from Singapore.bus_network._historical_project.codes.utils import create_graph_from_dataframe

data = bus_route.merge(bus_loc, on='bus_stop_code')

node_id_col = 'bus_stop_code'
order_col = 'order_no'
mline_id_col = ['bus_operator', 'bus_service_no', 'route_direction']
node_attr_list = ['bus_stop_code', 'x', 'y', 'lat', 'lon', 'road_name', 'stop_name']
edge_attr_list = ['bus_service_no', 'bus_operator', 'route_direction', 'stop_spacing']


# add nodes
# graph = graph_add_nodes_from_dataframe(graph, bus_loc, node_id_col, node_attr_list)
graph = create_graph_from_dataframe(
    data = data,
    node_id_col = node_id_col,
    order_col = order_col,
    mline_id_col = mline_id_col,
    node_attr_list = node_attr_list,
    edge_attr_list = edge_attr_list,
    space='l')

# processing edge attributes
# merge edge attributes, list -> string
for edge_attr in tqdm.tqdm(['bus_service_no', 'bus_operator', 'route_direction'], desc='Merge edge attributes...'):
    for edge in graph.edges():
        _edge_attr_val = list(set(graph.edges[edge][edge_attr]))
        _edge_attr_val.sort()
        _edge_attr_val = list(map(str, _edge_attr_val))
        graph.edges[edge][edge_attr] = ','.join(_edge_attr_val)

# merge edge attributes, 'stop_spacing'
for edge in graph.edges():
    graph.edges[edge]['stop_spacing'] = np.mean(graph.edges[edge]['stop_spacing'])


# save bus network graph
nx.write_gexf(graph, 'bus_network_2024-09-19.gexf')
nx.write_gml(graph, 'bus_network_2024-09-19.gml')

Merge edge attributes...: 100%|██████████| 3/3 [00:00<00:00, 40.99it/s]
